In [1]:

import functools
import openai

def _default_retries(cls):
    original = cls.__init__

    @functools.wraps(original)
    def __init__(self, *args, **kwargs):
        kwargs.setdefault("max_retries", 12)
        original(self, *args, **kwargs)

    cls.__init__ = __init__

for _cls in (openai.OpenAI, openai.AsyncOpenAI):
    _default_retries(_cls)


## Table of Contents
- [Setup and Prerequisites](#setup-and-prerequisites)
- [High-Level Overview](#high-level-overview)
- [Finding Posts with X Search](#finding-posts-with-x-search)
    - [Writing the Search Query](#writing-the-search-query)
- [Bringing Grok into the Equation](#bringing-grok-into-the-equation)
    - [Filtering Noisy Posts](#filtering-noisy-posts)
    - [Computing the Sentiment](#computing-the-sentiment)
- [Piecing it all Together](#piecing-it-all-together)
- [Conclusion](#conclusion)

## Setup and Prerequisites

All you need is an xAI API key, which you can grab over at the xAI [console](https://console.x.ai). Grok searches X for you, so you don't need an X developer account.

X Search is billed per post it fetches, on top of the usual token costs. See [tool invocation costs](https://docs.x.ai/developers/pricing#tool-invocation-costs) for details.

In [2]:
%pip install -q openai pydantic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import warnings

from dotenv import load_dotenv
from openai import OpenAI

# The OpenAI SDK doesn't know X Search's output types, so it warns when it serializes responses that include them.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")

load_dotenv()

client = OpenAI(
    base_url="https://api.x.ai/v1",
    api_key=os.getenv("XAI_API_KEY"),
)

## High-Level Overview

Our tool starts by asking Grok to search X for the latest bitcoin posts, using a search query we define upfront.

We send the posts it finds to `grok-4.3` with reasoning turned off. It can quickly classify which posts carry signal and are worth scoring, keeping those and discarding the rest. We then pass the high-signal posts to `grok-4.7` to leverage its reasoning capability to compute the sentiment score. Finally, we repeat the search every so often, and for each round that brings in new high-signal posts, we recalculate the score over the most recent ones. It's a rolling update, simple but effective.

This modular design pays off in a few ways. It gives us fine-grained control over each step, since searching, filtering and scoring get their own prompts, they can be tuned independently. If we want to tweak the filtering criteria or swap in a new xAI model later, it's a simple change, isolated from the sentiment process and vice-versa. Compare that to a single "super-prompt" approach: one tweak there could ripple across everything, muddying the results.

Another perk is how it plays to model strengths. That's where `grok-4.7`'s reasoning shines - as a [reasoning model](https://docs.x.ai/developers/model-capabilities/text/reasoning), it "thinks" before responding allowing for more analytical answers. By splitting the workload this way, we get efficiency and precision, tailored to each task.

More broadly, LLMs like Grok are a natural fit for this. Posts roll in from X in all sorts of languages—not just English—and Grok handles them effortlessly, no extra training needed. Plus, we can tweak classification or sentiment criteria on the fly by just updating the respective prompts, say, to catch different signals or refine scoring. Unlike traditional ML, where you'd retrain a model from scratch, this lets us experiment fast and iterate based on what we see.

The approach outlined above is just one way to do it; you could adjust how many posts you fetch or only recalculate sentiment once a bigger pile of new posts comes in, depending on your needs.

## Finding Posts with X Search

[X Search](https://docs.x.ai/developers/tools/x-search) is a built-in tool that lets Grok search X while it works on a request. We add it to the `tools` list, and Grok decides how to search, runs the searches on xAI's servers, and reads the results before it answers. There's no search code for us to write.

### Writing the Search Query
X Search understands X's advanced search operators, so we can describe the posts we want precisely. Here's the query we'll roll with:

In [4]:
SEARCH_QUERY = "(BTC OR Bitcoin OR #BTC OR #Bitcoin) min_faves:20 -is:retweet -is:reply"

This query grabs posts mentioning `BTC`, `Bitcoin`, `#BTC` or `#Bitcoin` that have at least 20 likes, excluding reposts and replies. Even with `grok-4.3` filtering low-signal posts later, this threshold helps cut down noise upfront: without it, the latest bitcoin posts are mostly spam and promotions.

To get the posts back in a form we can work with, we define them as a Pydantic model and use [structured outputs](https://docs.x.ai/developers/model-capabilities/text/structured-outputs) alongside the search tool. Finding and copying posts doesn't take much thought, so we use `grok-4.3` with a low reasoning effort to keep it fast.

Grok also cites every post it got from X Search by its URL. We use those citations to double-check the results, keeping only the posts that X Search actually returned.

In [5]:
import json
from datetime import datetime, timedelta, timezone

from pydantic import BaseModel, Field


class Post(BaseModel):
    post_id: str = Field(description="The numeric ID of the post, from its URL")
    username: str = Field(description="The author's X handle, without the @")
    text: str = Field(description="The full text of the post")
    created_at: str = Field(description="When the post was published, in ISO 8601 format")


class Posts(BaseModel):
    posts: list[Post]


def cited_post_ids(response) -> set[str]:
    return {
        annotation.url.rsplit("/", 1)[-1]
        for item in response.output
        if item.type == "message"
        for content in item.content
        for annotation in getattr(content, "annotations", None) or []
        if annotation.type == "url_citation" and "/status/" in annotation.url
    }


def find_posts(query: str, limit: int = 10) -> list[Post]:
    yesterday = (datetime.now(timezone.utc) - timedelta(days=1)).date().isoformat()
    response = client.responses.parse(
        model="grok-4.3",
        reasoning={"effort": "low"},
        input=(
            f"Search X for the latest posts matching this query: {query}\n"
            f"Return up to {limit} posts exactly as they appear in the search results. Don't make up or paraphrase posts."
        ),
        tools=[{"type": "x_search", "from_date": yesterday}],
        text_format=Posts,
    )

    cited_ids = cited_post_ids(response)
    return [post for post in response.output_parsed.posts if post.post_id in cited_ids]

Let's run a search:

In [6]:
posts = find_posts(SEARCH_QUERY)

print(f"Found {len(posts)} posts\n")
for post in posts:
    print(f"@{post.username} ({post.created_at}): {post.text}\n")

Found 10 posts

@IvanOnTech (2026-09-30T20:26:32Z): BITCOIN - FLYING HIGHER AND HIGHER!!!  ❤️🧡🧡💛🧡🧡💚

@Bitcoinmeraklsi (2026-09-30T20:26:10Z): #ETH 
Altcoinlerin fitilini ateşleyecek olan $ETH'un #BTC karşısındaki duruşu.

#Ethereum, $BTC karşısında güç kazanmaya başladığı zaman hem ETH daha agresif hareketler sergileyecek hemde peşi sıra altlar.

O yüzden ETH/BTC grafiğine baktığımızda gayet güçlü görünse de hala düşüş trendini kırmış değil.

Ne zaman ki bu trendi kırar o zaman güzel hareketler görmeye başlarız.

Takibe devam.

@Reis0766 (2026-09-30T20:26:06Z): Bu saate #bitcoin grafiğini Böyle takip eden kaç dostum var

@DarkoMatrix (2026-09-30T20:25:05Z): one thing i like about btc is that it doesn't need to be reinvented

it just needs better ways to participate

with strkbtc on @Starknet, btc can move through garden finance from bitcoin into starknet and connect with supported defi

the same btc can then be used for lending, borrowing, trading or supported staking opportunities

bu

Ok nice, the posts rolling in match our query's criteria, recent bitcoin mentions with some engagement, no reposts or replies. Next, we'll sift through them to filter out the noise.

## Bringing Grok into the Equation

### Filtering Noisy Posts
Our search query, despite its filters, grabs a lot - but not all of it's useful. Noise like ads, spam, or vague posts still creeps in. As laid out in the overview, we send the posts to `grok-4.3`. Its job? Quickly classify which ones have signal - posts that could sway sentiment—and which don't. We're not computing sentiment here, just picking the keepers to stash for later. `grok-4.3`'s speed, with reasoning turned off, makes it perfect for this, sifting through the batch in no time.

The prompt below tells `grok-4.3` to focus on post content—does it say something meaningful about Bitcoin market sentiment? It's just one way to write it; you could tweak it to weigh certain authors more, add new rules, trivially changing it in plain English. That's a perk of using LLMs here—no retraining like with classical ML, just update the prompt and go.

In [7]:
CLASSIFICATION_PROMPT = """You are an expert in bitcoin market sentiment analysis tasked with identifying X posts relevant to assessing bitcoin (BTC) market sentiment. Your goal is to analyze a list of posts, including their IDs and authors, and determine which posts contain information useful for bitcoin market sentiment analysis. You will return only the IDs of relevant posts.

To perform this task effectively, follow these instructions:

1. Examine the text content of each post to assess whether it discusses bitcoin (BTC) specifically in a way that could inform market sentiment analysis (e.g., opinions, trends, or observations about BTC's market behavior).
2. Consider who the author is to evaluate the post's potential influence or credibility, but prioritize the post content over the author.
3. Exclude posts that are purely promotional (e.g., ads, spam, or self-serving hype) or unrelated to bitcoin market sentiment (e.g., off-topic or generic cryptocurrency mentions without BTC focus).
4. For each post, determine if its content provides meaningful insight into BTC market sentiment, not whether the sentiment is positive or negative—your task is classification, not sentiment prediction.
5. Compile and return a list of post IDs that meet the criteria for relevance to BTC market sentiment analysis. If no posts qualify, return an empty list.

Key reminders:
- Focus solely on BTC-specific content, not general cryptocurrency discussions.
- Do not feel pressured to return IDs just because posts are provided; it's acceptable to return none if no posts are relevant.
- Avoid over-interpreting vague or over-hyped posts—relevance to market sentiment is the priority.
"""


class SignalPostIDs(BaseModel):
    post_ids: list[str]


def posts_json(posts: list[Post]) -> str:
    return json.dumps([post.model_dump() for post in posts], ensure_ascii=False, indent=2)


def filter_posts(posts: list[Post]) -> list[Post]:
    response = client.responses.parse(
        model="grok-4.3",
        reasoning={"effort": "none"},
        input=[
            {"role": "system", "content": CLASSIFICATION_PROMPT},
            {"role": "user", "content": f"Here are the posts: {posts_json(posts)}"},
        ],
        text_format=SignalPostIDs,
    )

    signal_ids = set(response.output_parsed.post_ids)
    return [post for post in posts if post.post_id in signal_ids]

In [8]:
signal_posts = filter_posts(posts)
noisy_posts = [post for post in posts if post not in signal_posts]

print("High Signal Posts")
for post in signal_posts:
    print(f"@{post.username}: {post.text}\n")
print("-" * 100)
print("Noisy Posts")
for post in noisy_posts:
    print(f"@{post.username}: {post.text}\n")

High Signal Posts
@IvanOnTech: BITCOIN - FLYING HIGHER AND HIGHER!!!  ❤️🧡🧡💛🧡🧡💚

@Reis0766: Bu saate #bitcoin grafiğini Böyle takip eden kaç dostum var

@astekz: $BTC - Why the f can I see us doing something like this between now and 2027

@BitcoinMagazine: JUST IN: 🇺🇸 U.S. Spot Bitcoin ETFs have seen 9 consecutive days of inflows totaling $3.07 billion 🚀

@pete_rizzo_: LIKE, IF YOU THINK #BITCOIN WILL HIT $100,000 AGAIN IN 2026

@NetAnalist: BTC Çakılmasına rağmen, coinim

Nasıl duruyor yüzde 4 Yukarıda ?

Giriş yerine Stop koydum bilginiz olsun

@JrKripto: Bu gece saat 03:00’te boğada mıyız değil miyiz belli olacak. 

Çok merak ediyorum saat 03:00’deki Tarihi Bitcoin aylık kapanışını kaç kişi bekliyor?? Bir yoklama alalım. 

Bitcoin, haftalık kapanışı geçen pazar 84.000$ üstü yaptı, aylık kapanışta 84.000$ üstü yapması durumunda boğa onayıdır.

Yeterli çoğunluk oluşursa JrKripto Mobil Uygulamamızdan canlı yayın açacağım.

App Store veya Google Play’de JrKripto yazıp aratıp indirin. Uy

The split above shows it's working: high-signal posts carry BTC market insights, while noisy ones are mostly promo, hype/spam or off-topic.

### Computing the Sentiment

With our high-signal posts filtered, we're ready to score Bitcoin's market sentiment. We hand them to `grok-4.7`, which digs into the text, weighs author influence, and outputs a score from -1 (bearish) to 1 (bullish), with 0 being neutral.

The prompt below tells Grok to analyze post content for sentiment, factor in author prominence, and average it into a final score. We use structured outputs again, so the score, Grok's reasoning, and the key posts that led to that score come back as separate fields. Again, this prompt is easily modifiable to add/remove instructions/criteria based on your requirements.

In [9]:
SENTIMENT_PROMPT = """You are an expert Bitcoin market analyst tasked with generating a sentiment score for Bitcoin (BTC) based exclusively on the provided X posts. Your role is to evaluate these posts and assign a sentiment score ranging from -1 (bearish) to 1 (bullish), with 0 indicating neutral, reflecting the overall market sentiment toward BTC.

To perform this task effectively, follow these numbered instructions:

1. Analyze the content of each X post to determine its sentiment toward Bitcoin—look for expressions of optimism (bullish), pessimism (bearish), or neutrality (e.g., factual statements without clear bias).
2. Evaluate whether the author is a prominent figure or a well-known organization in the crypto or finance industry, as this may amplify the post's credibility or impact.
3. Weigh each post's influence by combining the sentiment expressed in the content with the author's credibility—prioritize posts from influential authors with clear sentiment signals.
4. After reviewing all posts, calculate a final sentiment score (between -1 and 1) by averaging the weighted sentiments, rounding to one decimal place if needed.
5. Explain your conclusion in a concise reasoning paragraph, and list the key posts with brief notes on their influence.

Key reminders:
- Base your analysis solely on the provided X posts—do not use external data.
- Focus on sentiment specific to Bitcoin (BTC), not cryptocurrencies in general.
- Ensure your reasoning is concise yet clearly justifies the score and key post selections.
"""


class KeyPost(BaseModel):
    post_id: str
    note: str = Field(description="Why this post influenced the score")


class Sentiment(BaseModel):
    score: float = Field(description="From -1 (bearish) to 1 (bullish), with 0 being neutral")
    reasoning: str = Field(description="A brief explanation of how you arrived at the score")
    key_posts: list[KeyPost]


def score_sentiment(posts: list[Post]) -> tuple[Sentiment, str]:
    response = client.responses.parse(
        model="grok-4.7",
        input=[
            {"role": "system", "content": SENTIMENT_PROMPT},
            {"role": "user", "content": f"Here are the posts: {posts_json(posts)}"},
        ],
        text_format=Sentiment,
    )

    reasoning_trace = "\n".join(
        part.text
        for item in response.output
        if item.type == "reasoning"
        for part in item.content or []
    )
    return response.output_parsed, reasoning_trace

In [10]:
sentiment, reasoning_trace = score_sentiment(signal_posts)

print(f"BTC Sentiment: {sentiment.score}")
print(f"Reasoning: {sentiment.reasoning}")
print("Key posts:")
for key_post in sentiment.key_posts:
    print(f"- https://x.com/i/status/{key_post.post_id}: {key_post.note}")

BTC Sentiment: 0.7
Reasoning: Prominent voices dominate with clear bullish signals: IvanOnTech celebrates BTC flying higher, BitcoinMagazine highlights massive consecutive ETF inflows, and Pete Rizzo polls for a return to $100k. These outweigh milder or mixed posts about chart watching, potential monthly closes, and one mention of a short-term dip, producing a solidly bullish overall sentiment.
Key posts:
- https://x.com/i/status/2105392417838903672: BitcoinMagazine (high-credibility outlet) reports 9 straight days of US spot BTC ETF inflows totaling $3.07B with rocket emoji—strong institutional bullish signal.
- https://x.com/i/status/2105393881466692076: IvanOnTech, a known crypto influencer, exuberantly states Bitcoin is flying higher and higher—direct high-impact bullish sentiment.
- https://x.com/i/status/2105391445473468535: Pete Rizzo, established Bitcoin commentator, invites likes for BTC hitting $100k again in 2026, reinforcing upside price expectations.


What's cool here? Beyond the final sentiment result, we can peek at Grok's reasoning trace, think of this as Grok's thought process that it went through before coming up with its final sentiment score. This can be very valuable when you're looking to debug or edit your prompt; seeing which paths or choices the model is making means you can make more targeted prompt modifications to achieve your desired behavior.

In the trace below, you can see Grok translate the posts that aren't in English, score each post on its own, and give the posts from prominent accounts more weight when it averages them into the final score.

In [11]:
print(reasoning_trace)

The user wants me to generate a sentiment score for Bitcoin based on the provided X posts. I need to output JSON matching the schema.

Let me analyze each post:

1. **IvanOnTech** (post_id: 2105393881466692076): "BITCOIN - FLYING HIGHER AND HIGHER!!! ❤️🧡🧡💛🧡🧡💚"
   - Very bullish. IvanOnTech is a known crypto YouTuber/influencer. High credibility.
   - Sentiment: Strongly bullish (+0.9 or so)

2. **Reis0766** (post_id: 2105393772020510759): "Bu saate #bitcoin grafiğini Böyle takip eden kaç dostum var" (Turkish: "How many friends are there watching the #bitcoin chart at this hour like this")
   - Neutral, just asking who's watching the chart. No clear sentiment.
   - Low influence unknown user.
   - Sentiment: Neutral (0)

3. **astekz** (post_id: 2105392457399627783): "$BTC - Why the f can I see us doing something like this between now and 2027"
   - Ambiguous without the image, but "why the f can I see us doing something like this" could be referring to a bullish chart projection. The ph

## Piecing it all Together

This is where it all comes together—search, filter, and score, round after round. Every `interval_seconds`, the function below searches X again and skips posts it has already seen. `grok-4.3` classifies the new ones, and whenever new high-signal posts come in, `grok-4.7` rescores the sentiment over the most recent `window` of them. Run the cell below to watch the sentiment update as new posts come in.

In [12]:
import time

from IPython.display import clear_output


def track_sentiment(query: str, rounds: int = 3, interval_seconds: int = 60, window: int = 20):
    seen_ids: set[str] = set()
    signal_posts: list[Post] = []
    sentiment = None

    for round_number in range(1, rounds + 1):
        new_posts = [post for post in find_posts(query) if post.post_id not in seen_ids]
        seen_ids.update(post.post_id for post in new_posts)
        new_signal_posts = filter_posts(new_posts) if new_posts else []
        signal_posts.extend(new_signal_posts)

        if new_signal_posts:
            sentiment, _ = score_sentiment(signal_posts[-window:])

        clear_output(wait=True)
        print(f"Round {round_number} of {rounds}: {len(new_posts)} new posts, {len(new_signal_posts)} with signal")
        print(f"Scored {min(len(signal_posts), window)} of {len(signal_posts)} high-signal posts so far")
        if sentiment:
            print(f"\nBTC Sentiment: {sentiment.score}\nReasoning: {sentiment.reasoning}")

        if round_number < rounds:
            time.sleep(interval_seconds)

In [13]:
track_sentiment(SEARCH_QUERY)

Round 3 of 3: 1 new posts, 1 with signal
Scored 5 of 5 high-signal posts so far

BTC Sentiment: 0.7
Reasoning: All five posts express bullish sentiment toward BTC, ranging from cautious technical optimism to strong buy advocacy and positive institutional news. The highest-impact signal is BitcoinMagazine's report of sustained ETF inflows, amplified by the outlet's prominence, while trader and retail posts reinforce a preference for longs and accumulation. No bearish views appear, yielding a clearly positive but not euphoric aggregate.


## Conclusion

We've built a tool that pulls the latest Bitcoin posts with Grok's X Search tool, filters noise with `grok-4.3`, and scores sentiment with `grok-4.7`'s reasoning. It's a lean, composable system—each piece (searching, filtering, scoring) slots together to deliver live market insights, and it only needs an xAI API key.

Here's what we've learned:
- A modular setup can often be favorable to a monolith, swap models or tune parts without breaking the whole.
- Match tasks to model strengths, `grok-4.3`'s speed for searching and filtering and `grok-4.7`'s reasoning for sentiment.
- Built-in tools like X Search let Grok gather live data for you, and citations let you check exactly where it came from.
- Prompts are flexible—tweak them in plain English to shift focus and adjust behavior according to your needs.
- LLMs like Grok thrive here: multilingual, adaptable, and quick to iterate.

Want to push it further? Turn on `enable_image_understanding` in the X Search tool so Grok can read the charts people post, or use `allowed_x_handles` to follow a list of analysts you trust. Or go dynamic: have Grok write the search query itself based on real-world events (e.g., "BTC crash" after a dip). Experiment, tweak, and see where it takes you.